# Sincronizar notebooks e scripts do GitHub para o Google Drive

Execute as células **1 a 5** no Colab conectado ao VS Code. Não é necessário GPU e nenhum treinamento é iniciado.

O notebook busca a branch privada `main`, fixa um commit e copia **todos os arquivos `.ipynb` e `.py` desse commit** para `/content/drive/MyDrive/TCC_Final`, mantendo as pastas relativas (`Notebooks/`, `experimentos/`, `tests/` etc.). Arquivos idênticos são ignorados; versões diferentes já existentes no Drive são preservadas em `codigo_historico/` antes da atualização. O manifesto fica em `codigo_sync/<commit>.json`.

Primeiro faça commit e push das alterações locais que devem entrar no espelho. Este notebook não acessa o disco `E:` e não copia datasets, checkpoints, resultados ou credenciais para o GitHub. Não exiba a senha ou o token.

In [6]:
# 1. Dependência de criptografia; GPU não é necessária.
%pip -q install cryptography
print('Dependência disponível; nenhum treino será iniciado.')

Dependência disponível; nenhum treino será iniciado.


In [7]:
# 2. Montar o Drive na mesma conta que guarda TCC_Final.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
ROOT.mkdir(parents=True, exist_ok=True)
print('Pasta do projeto no Drive:', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pasta do projeto no Drive: /content/drive/MyDrive/TCC_Final


In [8]:
# 3. Desbloquear a credencial GitHub existente, sem imprimir token ou senha.
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1:
        raise ValueError('Versão da credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000:
    raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(
    algorithm=hashes.SHA256(), length=32,
    salt=base64.b64decode(credential['salt']), iterations=iterations,
).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada somente nesta sessão.')

Credencial desbloqueada somente nesta sessão.


In [9]:
# 4. Baixar um snapshot imutável da branch privada para /content.
import io, re, shutil, stat, tempfile, time, urllib.error, urllib.request, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'main'
def github_read(url):
    request = urllib.request.Request(
        url, headers={'Authorization': f'Bearer {token}',
                      'Accept': 'application/vnd.github+json',
                      'User-Agent': 'TCC-Codigo-Drive-Colab'})
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            chunks, total, last_notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block)
                total += len(block)
                if time.monotonic() - last_notice >= 30:
                    print(f'Snapshot: {total / 1048576:.1f} MiB recebidos', flush=True)
                    last_notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira acesso ao repositório e à branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    if not re.fullmatch(r'[0-9a-f]{40}', commit):
        raise ValueError('SHA de commit inválido')
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/espelhar_codigo_drive.py',
                'Notebooks/fase_3/fase3_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do commit', commit, flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc_codigo_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2:
                        continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts:
                        raise ValueError('Caminho inseguro no snapshot')
                    if stat.S_IFMT(item.external_attr >> 16) == stat.S_IFLNK:
                        raise ValueError('Link simbólico no snapshot')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output:
                        shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique o script de espelho na branch antes de executar')
            staged.rename(CODE_ROOT)
    if not all((CODE_ROOT / name).is_file() for name in required):
        raise FileNotFoundError('Snapshot incompleto; use um runtime novo')
finally:
    del token
print('Código do commit disponível em:', CODE_ROOT)

Baixando snapshot do commit 7f74e70574fb82f7a00500fc1549c4a21cc48583
Código do commit disponível em: /content/tcc_codigo/7f74e70574fb82f7a00500fc1549c4a21cc48583


In [10]:
# 5. Espelhar todos os .ipynb/.py; atualizações conservam cópias anteriores.
import subprocess, sys
subprocess.run(
    [sys.executable, '-u', str(CODE_ROOT / 'experimentos/espelhar_codigo_drive.py'),
     '--source', str(CODE_ROOT), '--drive-root', str(ROOT), '--commit', commit],
    check=True,
)
manifest_path = ROOT / 'codigo_sync' / f'{commit}.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(f"Conferido: {manifest['file_count']} notebooks/scripts; "
      f"{manifest['total_bytes'] / 1e6:.2f} MB; commit {manifest['commit']}")
print('Exemplos de destinos:')
for relative in ('Notebooks/fase_3/fase3_9/rodar_fase3_9_tgif_colab.ipynb',
                 'Notebooks/fase_3/fase3_9/fase3_9_tgif_finetune.py',
                 'experimentos/espelhar_codigo_drive.py'):
    print(' ', ROOT / relative, '| presente:', (ROOT / relative).is_file())
print('Manifesto:', manifest_path)

Conferido: 83 notebooks/scripts; 6.94 MB; commit 7f74e70574fb82f7a00500fc1549c4a21cc48583
Exemplos de destinos:
  /content/drive/MyDrive/TCC_Final/Notebooks/fase_3/rodar_fase3_9_tgif_colab.ipynb | presente: True
  /content/drive/MyDrive/TCC_Final/experimentos/fase3_9_tgif_finetune.py | presente: True
  /content/drive/MyDrive/TCC_Final/experimentos/espelhar_codigo_drive.py | presente: True
Manifesto: /content/drive/MyDrive/TCC_Final/codigo_sync/7f74e70574fb82f7a00500fc1549c4a21cc48583.json
